In [55]:
# loding the librarys
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [56]:
# loding the dataset
df=pd.read_csv('P:\Internship_Final\Datasets\Amazon_cleaned_dataset.csv')

In [57]:
# making a copy
df1=df.copy()

##### first we will combine all the text features

In [58]:
# Create a combined text feature
df1["combined_text"] = (df1["name"].astype(str) + " " +df1["main_category"].astype(str) + " " +df1["sub_category"].astype(str))

- The recommendation system needs to understand the overall meaning of each product,
- not just individual columns. The product name, main category, and sub category each contain useful information,
- but when combined, they provide a more complete description of the product.

In [59]:
df1[["combined_text"]].head()

,combined_text
0,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...
1,LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...
2,LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...
3,LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...
4,Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...


##### Now we will create the discount persentage

In [80]:
# Create discount percentage column safely
df1["discount_percentage"] = np.where(df1["actual_price"] > 0,((df1["actual_price"] - df1["discount_price"]) / df1["actual_price"]) * 100,0)
df1["discount_percentage"].describe()

count    551585.000000
mean         10.542453
std           7.911245
min        -305.240410
25%           5.710543
50%          10.047349
75%          14.839726
max          73.965466
Name: discount_percentage, dtype: float64

- using only actual_price and discount_price doesn't tell us how much discount a customer is getting.
- thats why we created the discountedd persentage

#####  Now we will check the  Skewness of Numerical Features

In [61]:
# chcking the skewness
df1[["ratings","no_of_ratings","discount_price","actual_price","discount_percentage"]].skew()

ratings                -1.637230
no_of_ratings          40.561040
discount_price         17.654710
actual_price           22.418793
discount_percentage   -77.365616
dtype: float64

##### Observation
- The no_of_ratings, discount_price, and actual_price features are highly positively skewed, indicating the presence of a few products with extremely high values.
- The ratings feature is slightly negatively skewed, which is expected because ratings are limited to a range of 1 to 5.
- These skewed distributions suggest that some numerical features may require transformation before being used in the recommendation model.

##### Now we will transfrom the skew numarical columns 

In [62]:
# now we will apply the log transformation.
df1["no_of_ratings"] = np.log1p(df1["no_of_ratings"])
df1["discount_price"] = np.log1p(df1["discount_price"])
df1["actual_price"] = np.log1p(df1["actual_price"])

In [63]:
# after the log transfrom
df1[["no_of_ratings","discount_price","actual_price"]].skew()

no_of_ratings     1.164349
discount_price    1.410621
actual_price      0.890343
dtype: float64

##### Now we will convert the text feature into numaric feature

In [64]:
# importing  the library
from sentence_transformers import SentenceTransformer

In [65]:
# Now we will lode the pre train sentance tranfromar model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [66]:
# Now we will make the text embeddinhg
embeddings = model.encode(df1["combined_text"].tolist(),batch_size=64,show_progress_bar=True,convert_to_numpy=True)

Batches:   0%|          | 0/8619 [00:00<?, ?it/s]

##### Now we will scale the numaric feature

In [67]:
# import the standard scaler
from sklearn.preprocessing import StandardScaler

In [68]:
# selecting the numaric feature 
numerical_features = ["ratings","no_of_ratings","discount_price","actual_price","discount_percentage"]

In [69]:
# Verify that all numerical features contain only finite values
np.isfinite(df1[numerical_features]).all()

ratings                True
no_of_ratings          True
discount_price         True
actual_price           True
discount_percentage    True
dtype: bool

In [70]:
# Check for missing values in the numerical features
df1[numerical_features].isnull().sum()

ratings                0
no_of_ratings          0
discount_price         0
actual_price           0
discount_percentage    0
dtype: int64

In [71]:
# Create the scaler and scale the numerical features
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df1[numerical_features])

# Verify scaling output
print("NaN values in scaled_features:", np.isnan(scaled_features).sum())

NaN values in scaled_features: 0


##### Now we will combine the Text Embeddings and Numerical Features

In [72]:
# we are using this library so that we can horizontally combines multiple sparse matrices into a single sparse matrix.
from scipy.sparse import hstack

In [73]:

# Now we will combine text embeddings and scaled numerical features
hybrid_features = np.hstack((embeddings, scaled_features))

In [74]:
# Saving the feature-engineered dataset
df1.to_csv("../Datasets/Amazon_feature_dataset.csv", index=False)

##### Now we will save all the models 

In [75]:
import joblib

In [76]:
# first we will save the text embadding
joblib.dump(embeddings, "../Models/text_embeddings.pkl")

['../Models/text_embeddings.pkl']

In [77]:
# now we will save the scalled featured models
joblib.dump(scaled_features, "../Models/scaled_features.pkl")

['../Models/scaled_features.pkl']

In [78]:
# now we will save the scaler
joblib.dump(scaler, "../Models/scaler.pkl")

['../Models/scaler.pkl']

In [81]:
# now we will save the hybrid feature model
joblib.dump(hybrid_features, "../Models/hybrid_features.pkl")
print("KNN model saved successfully")

KNN model saved successfully


##### Overall Summary – Feature Engineering
- Successfully engineered new features to improve the quality of the recommendation system.
- Combined product name, main category, and sub-category into a single text feature for richer semantic representation.
- Generated text embeddings using the Sentence Transformer model to capture product similarity.
- Created a new numerical feature, discount_percentage, and handled invalid values to ensure data quality.
- Applied log transformation to reduce skewness in numerical features.
- Standardized numerical features using StandardScaler to bring them to a common scale.
- Combined text embeddings and scaled numerical features into a hybrid feature matrix, creating a comprehensive representation of each product.
- The final hybrid feature matrix is clean, free from missing values, and ready for training the recommendation model.